# Day 0 Spike — BITE 추론 전용 (GPU 시간 최소화)

**이 노트북은 Colab GPU에서 돌리는 용도.** 시각화/분석은 로컬에서.

흐름: 설치 → 추론 → 결과(.pkl) 저장 → Drive 업로드 → 종료

⏱ 예상 시간: 설치 5분 + 이미지당 3-5분

In [ ]:
# === [1/6] 환경 확인 + Drive 마운트 ===
import torch, os, sys, time
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요!"
print(f"GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_mem/1e9:.1f}GB")

from google.colab import drive
drive.mount('/content/drive')

# 결과 저장 경로 (Drive에 저장 → Colab 끝나도 유지)
SAVE_DIR = '/content/drive/MyDrive/dog-measurement-spike'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"결과 저장: {SAVE_DIR}")

In [ ]:
# === [2/6] BITE 클론 + 의존성 설치 ===
t0 = time.time()

!git lfs install
!git clone https://huggingface.co/spaces/runa91/bite_gradio /content/bite_release

!pip install -q trimesh scipy matplotlib opencv-python-headless chumpy openpyxl dominate kornia
!pip install -q git+https://github.com/runa91/FrEIA.git

# pytorch3d (Colab용)
pyt_ver = torch.__version__.split('+')[0].replace('.', '')
cuda_ver = torch.version.cuda.replace('.', '')
whl_url = f"https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py3{sys.version_info.minor}_cu{cuda_ver}_pyt{pyt_ver}/download.html"
!pip install -q --no-index --no-cache-dir pytorch3d -f {whl_url} || echo "pytorch3d wheel 없음 — 소스 빌드 시도"

# pytorch3d wheel 실패 시 소스 빌드
try:
    import pytorch3d
    print(f"pytorch3d {pytorch3d.__version__} OK")
except ImportError:
    print("pytorch3d 소스 빌드 중... (5-10분)")
    !pip install -q 'git+https://github.com/facebookresearch/pytorch3d.git'

print(f"\n설치 완료: {time.time()-t0:.0f}초")

In [ ]:
# === [3/6] BITE 모델 로드 ===
BITE_DIR = '/content/bite_release'
sys.path.insert(0, BITE_DIR)
sys.path.insert(0, os.path.join(BITE_DIR, 'src'))

# SMAL 로드 테스트
try:
    from src.smal_pytorch.smal_model.smal_torch_new import SMAL
    print("[OK] SMAL import")
except Exception as e:
    print(f"[FAIL] {e}")
    raise

# BITE 추론 함수 import
try:
    from scripts.gradio_demo import run_bite_inference, run_bbox_inference
    print("[OK] BITE inference functions")
except Exception as e:
    print(f"[WARN] gradio_demo import 실패: {e}")
    print("직접 파이프라인을 구성합니다...")

In [ ]:
# === [4/6] 테스트 이미지 준비 ===
import glob
from PIL import Image

# 방법 1: BITE에 포함된 예제 이미지
test_images = sorted(glob.glob(os.path.join(BITE_DIR, 'datasets', 'test_image_crops', '*')))

# 방법 2: Drive에서 이미지 로드
drive_images = sorted(glob.glob('/content/drive/MyDrive/dog-measurement-spike/input_images/*'))
if drive_images:
    test_images = drive_images
    print(f"Drive에서 이미지 {len(test_images)}장 로드")

# 방법 3: 직접 업로드
if not test_images:
    from google.colab import files
    uploaded = files.upload()
    test_images = list(uploaded.keys())

print(f"테스트 이미지 {len(test_images)}장:")
for p in test_images[:5]:
    print(f"  {os.path.basename(p)}")

In [ ]:
# === [5/6] 추론 실행 + 결과 저장 ===
import pickle
import numpy as np
import trimesh

results = []

for idx, img_path in enumerate(test_images):
    name = os.path.splitext(os.path.basename(img_path))[0]
    print(f"\n[{idx+1}/{len(test_images)}] {name}")
    t0 = time.time()

    try:
        # Bbox 검출
        _, bbox = run_bbox_inference(img_path)
        print(f"  bbox: {bbox}")

        # BITE 추론 (test-time optimization)
        result = run_bite_inference(
            img_path, bbox=bbox,
            apply_ttopt=True,
            dog_name=name
        )

        elapsed = time.time() - t0
        print(f"  완료: {elapsed:.0f}초")

        # 메시 파일 찾기 + 저장
        mesh_files = glob.glob(f"/content/bite_release/**/results_gradio/**/{name}*", recursive=True)
        if mesh_files:
            for mf in mesh_files:
                ext = os.path.splitext(mf)[1]
                dest = os.path.join(SAVE_DIR, f"{name}{ext}")
                import shutil
                shutil.copy2(mf, dest)
                print(f"  저장: {dest}")

            # 메시 로드해서 vertex/face 데이터도 pkl로 저장
            for mf in mesh_files:
                if mf.endswith('.obj') or mf.endswith('.glb'):
                    mesh = trimesh.load(mf)
                    results.append({
                        'name': name,
                        'vertices': np.array(mesh.vertices),
                        'faces': np.array(mesh.faces),
                        'image_path': img_path,
                        'elapsed_sec': elapsed,
                    })
                    break

        print(f"  [OK]")

    except Exception as e:
        print(f"  [FAIL] {e}")
        results.append({
            'name': name,
            'error': str(e),
            'image_path': img_path,
        })

# 전체 결과를 pkl로 저장
pkl_path = os.path.join(SAVE_DIR, 'spike_results.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(results, f)
print(f"\n=== 전체 결과 저장: {pkl_path} ===")
print(f"성공: {sum(1 for r in results if 'vertices' in r)} / {len(results)}")

In [ ]:
# === [6/6] SMAL 모델 데이터도 Drive에 백업 (로컬 분석용) ===
import shutil

smal_backup = os.path.join(SAVE_DIR, 'smal_model')
os.makedirs(smal_backup, exist_ok=True)

smal_files = [
    'data/smal_data/new_dog_models/my_smpl_39dogsnorm_Jr_4_dog.pkl',
    'data/smal_data/new_dog_models/my_smpl_data_39dogsnorm_Jr_4_dog.pkl',
]
for sf in smal_files:
    src = os.path.join(BITE_DIR, sf)
    if os.path.exists(src):
        shutil.copy2(src, smal_backup)
        print(f"백업: {os.path.basename(sf)}")

print(f"\n=== Colab 작업 완료 ===")
print(f"Drive 경로: {SAVE_DIR}")
print(f"로컬에서 할 일:")
print(f"  1. spike_results.pkl 다운로드")
print(f"  2. python scripts/analyze_spike.py 실행")
print(f"  3. Pass/Fail 판정")